# CrashDefNet quick start

<a href="https://colab.research.google.com/github/stofe94/crash-def-net/blob/main/examples/quick_start/crashdefnet-quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A graph neural network learns from finite element simulations how a plate deforms when an
actuator presses into it. Then it predicts this deformation on its own, time step by time step,
for plates it has never seen.

This notebook runs the whole workflow on a small example on Colab's free GPU, in under
10 minutes: set up, prepare the data, train, predict, show the result. The model trains for a
few minutes on 8 plates: the absolute minimum to show that the workflow runs, not a usable
result.

**Start:** *Runtime → Run all*. If Colab warns that the notebook was not authored by Google,
choose *Run anyway*.

## 1. Set up (about 2 minutes)

Downloads CrashDefNet, installs the packages Colab is missing and fetches the test file of
DeepMind's `deforming_plate` dataset (0.8 GB, 100 simulated plates).

In [ ]:
import torch

%cd /content
![ -d crash-def-net ] || git clone -q --depth 1 https://github.com/stofe94/crash-def-net.git
%cd crash-def-net
!pip install -q torch-geometric tfrecord toml pyvista
!apt-get install -y -qq libegl1 libegl-mesa0 libgl1-mesa-dri > /dev/null   # draws the animation
%env VTK_DEFAULT_OPENGL_WINDOW=vtkEGLRenderWindow
url = "https://storage.googleapis.com/dm-meshgraphnets/deforming_plate"
!wget -q -nc -P datasets/deforming_plate/raw_dataset {url}/meta.json {url}/test.tfrecord
print("GPU:", torch.cuda.get_device_name() if torch.cuda.is_available()
      else "none, training runs on the CPU and takes longer")

## 2. Prepare the data (a few seconds)

Turns 12 of the plates into graphs: the nodes of the finite element mesh, connected along the
mesh and where the actuator touches the plate. 8 plates are for training, 2 check the model
during training, and 2 it never sees: it predicts those at the end.

In [ ]:
!python -m dataset_preprocessor.export

## 3. Train (about 3 minutes)

The model is MGN-T (MeshGraphNet-Transformer) in a small size. From one time step it learns to
predict the next: how each node moves and which stress it carries. After every epoch it predicts
the 2 check plates on its own, and the epoch that does this best is kept.

In [ ]:
!python train/train.py --config examples/quick_start/train.toml --epochs 20

## 4. Predict (under a minute)

The trained model predicts the 2 unseen plates from the first time step to the last, each step
from its own previous prediction, and compares them with the simulation.

In [ ]:
!python predict/rollout.py   # the model of step 3 on the 2 unseen plates

## 5. Result

The 2 plates the model has never seen: left the simulation (finite element method), right the
prediction of the model. Colour: von Mises stress.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

rollout = sorted(Path("rollouts").iterdir())[-1]   # the newest prediction
!python utils/animate_rollout.py {rollout} --trajectory test_traj0010 --output plate_1.gif
!python utils/animate_rollout.py {rollout} --trajectory test_traj0011 --output plate_2.gif
display(Image("plate_1.gif"), Image("plate_2.gif"))

**Not a usable result:** this notebook is the absolute minimum to show that the workflow runs
and has nothing to do with a real training, which is what leads to usable results (see the
[documentation](https://stofe94.github.io/crash-def-net/results/deforming-plate/)).

## Next steps

Installation on your own machine, your own data, the other architecture (MeshGraphNets):
[documentation](https://stofe94.github.io/crash-def-net/)